# Baseline Linear Regression

training linear regression in blocks using baseline features + traficom features

## 1. Imports

In [1]:
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

## 2. Load data

In [2]:
train_path = "../../datasets/splits/train_grouped.csv"
validation_path = "../../datasets/splits/validation_grouped.csv"

In [3]:
train_df = pd.read_csv(train_path)
validation_df = pd.read_csv(validation_path)

## 3. Quick data checks

In [4]:
print("Train shape:", train_df.shape)
print("Validation shape:", validation_df.shape)

Train shape: (7954, 79)
Validation shape: (1689, 79)


In [5]:
print("Train info")
train_df.info()

print("\nValidation info")
validation_df.info()

Train info
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7954 entries, 0 to 7953
Data columns (total 79 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   product_id                      7954 non-null   int64  
 1   part_name                       7954 non-null   object 
 2   price                           7954 non-null   float64
 3   quality_grade                   7954 non-null   object 
 4   oem_number                      7954 non-null   object 
 5   mileage                         7954 non-null   float64
 6   brand                           7954 non-null   object 
 7   model                           7954 non-null   object 
 8   category                        7954 non-null   object 
 9   subcategory                     7954 non-null   object 
 10  scrape_date                     7954 non-null   object 
 11  year_start                      7954 non-null   int64  
 12  year_end               

## 4. Define feature groups

Registry lifecycle features describe vehicle registration-history patterns from Traficom data. Listing dynamics features describe scrape-window behavior of listings.

In [6]:
baseline_features = [
    "part_name",
    "quality_grade",
    "oem_number",
    "mileage",
    "brand",
    "model",
    "category",
    "subcategory",
    "year_start",
    "year_end",
    "year_span",
    "year_mid",
    "repair_status",
]

traficom_features = [
    "model_total_registered",
    "model_median_vehicle_age",
    "model_mean_vehicle_age",
    "model_median_mileage",
    "model_mean_mileage",
    "model_median_engine_cc",
    "model_median_power_kw",
    "model_median_mass_kg",
    "brand_total_registered",
    "brand_median_vehicle_age",
    "brand_mean_vehicle_age",
    "brand_median_mileage",
    "brand_mean_mileage",
]

# Registry lifecycle features describe vehicle registration-history features.
registry_lifecycle_candidates = [
    "model_firstreg_total_2014_2026",
    "model_firstreg_recent_share",
    "model_firstreg_old_share",
    "model_firstreg_weighted_year",
    "model_firstreg_peak_year",
    "model_firstreg_peak_count",
    "model_firstreg_year_span",
    "brand_firstreg_total_2014_2026",
    "brand_firstreg_recent_share",
    "brand_firstreg_old_share",
    "brand_firstreg_weighted_year",
    "brand_firstreg_peak_year",
    "brand_firstreg_peak_count",
    "brand_firstreg_year_span",
]

registry_lifecycle_features = [
    column for column in registry_lifecycle_candidates
    if column in train_df.columns
]

missing_registry_lifecycle_features = [
    column for column in registry_lifecycle_candidates
    if column not in train_df.columns
]

traficom_extended_candidates = [
    "model_share_of_market",
    "model_share_within_brand",
    "model_share_over_10y",
    "model_share_over_200k_km",
    "model_automatic_share",
    "model_petrol_share",
    "model_diesel_share",
    "model_ev_share",
    "model_hybrid_share",
    "model_turbo_share",
    "brand_median_engine_cc",
    "brand_median_power_kw",
    "brand_median_mass_kg",
    "brand_share_of_market",
    "brand_share_over_10y",
    "brand_share_over_200k_km",
    "brand_automatic_share",
    "brand_petrol_share",
    "brand_diesel_share",
    "brand_ev_share",
    "brand_hybrid_share",
    "brand_turbo_share",
]

traficom_extended_features = [
    column for column in traficom_extended_candidates if column in train_df.columns
]

missing_traficom_extended_features = [
    column for column in traficom_extended_candidates if column not in train_df.columns
]

print("Found registry lifecycle features:")
print(registry_lifecycle_features)

print("\nMissing registry lifecycle features:")
print(missing_registry_lifecycle_features)

print("\nFound extended Traficom features:")
print(traficom_extended_features)

print("\nMissing extended Traficom features:")
print(missing_traficom_extended_features)

# Listing dynamics features describe scrape-window listing behavior,
# not registry lifecycle or vehicle registration-history features.
listing_dynamics_features = [
    "times_observed",
    "observed_span_days",
    "price_changed_flag",
    "price_change_count",
    "absolute_price_change",
    "relative_price_change_pct",
]

linear_trusted_selection_exclusion_reasons = {
    "times_observed": "Uses the full listing observation count, which is only known after the listing lifecycle has unfolded.",
    "observed_span_days": "Uses the full observed listing span, which reaches into the future for earlier snapshots.",
    "price_changed_flag": "Summarizes whether the listing price ever changed across the whole listing history.",
    "price_change_count": "Counts price changes over the full listing history, including future changes.",
    "absolute_price_change": "Uses the total absolute price movement across the complete listing history.",
    "relative_price_change_pct": "Uses the net percentage price movement across the complete listing history.",
    "first_seen_date": "Raw datetime fields are omitted from the trusted linear comparison to keep the feature space clean.",
    "last_seen_date": "Raw datetime fields are omitted from the trusted linear comparison and the end date is future-looking.",
}
linear_trusted_selection_exclusions = set(linear_trusted_selection_exclusion_reasons)

# Keep this alias so the existing notebook cells continue to run.
lifecycle_features = listing_dynamics_features

recommended_inclusion_reasons = {
    "first_seen_date": "Observed listing start within the crawl window; retained only for exploratory comparisons.",
    "last_seen_date": "Observed listing end within the crawl window; retained only for exploratory comparisons.",
    "brand_is_known_model_family": "Low-cardinality metadata flag; slight validation improvement when included.",
    "mileage_missing_flag": "Tracks rows where mileage was originally missing before hierarchical median imputation.",
}

recommended_exclusion_reasons = {
    "product_id": "High-cardinality listing identifier; encourages memorization and hurt MAE.",
    "scrape_date": "Current crawl wave rather than part value; worsened validation when included.",
    "brand_merge_key": "Redundant normalized brand key that overlaps with brand.",
    "model_merge_key": "Redundant normalized model key that overlaps with model.",
    "model_family_clean": "Collapsed model family duplicate of the existing model labels.",
    "model_looks_like_part_taxonomy": "Constant in the training split, so it adds no signal.",
}

manual_all_feature_groups = list(dict.fromkeys(
    baseline_features
    + traficom_features
    + registry_lifecycle_features
    + traficom_extended_features
    + listing_dynamics_features
))

recommended_excluded_features = set(recommended_exclusion_reasons)

recommended_model_features = [
    column for column in train_df.columns
    if column != "price" and column not in recommended_excluded_features
]

manual_all_feature_groups_leakage_safe = [
    column for column in manual_all_feature_groups
    if column not in linear_trusted_selection_exclusions
]

recommended_model_features_leakage_safe = [
    column for column in recommended_model_features
    if column not in linear_trusted_selection_exclusions
]

missing_from_previous_manual_all = [
    column for column in recommended_model_features if column not in manual_all_feature_groups
]

feature_audit_df = pd.DataFrame(
    [
        {
            "feature": column,
            "status": "missing_from_previous_manual_all",
            "reason": recommended_inclusion_reasons.get(
                column,
                "New candidate feature found in the dataset and included in the recommended model feature set.",
            ),
        }
        for column in missing_from_previous_manual_all
    ]
    + [
        {
            "feature": column,
            "status": "recommended_exclusion",
            "reason": recommended_exclusion_reasons[column],
        }
        for column in sorted(recommended_excluded_features)
    ]
    + [
        {
            "feature": column,
            "status": "trusted_selection_exclusion",
            "reason": linear_trusted_selection_exclusion_reasons[column],
        }
        for column in sorted(linear_trusted_selection_exclusions)
    ]
)

print("Columns missing from the previous manual all-features set:")
print(missing_from_previous_manual_all)

print("\nRecommended exclusions:")
print(sorted(recommended_excluded_features))

print("\nFeatures excluded from trusted linear selection:")
print(sorted(linear_trusted_selection_exclusions))

print("\nNumber of recommended model features:", len(recommended_model_features))
print("Number of trusted recommended model features:", len(recommended_model_features_leakage_safe))

feature_audit_df

Found registry lifecycle features:
['model_firstreg_total_2014_2026', 'model_firstreg_recent_share', 'model_firstreg_old_share', 'model_firstreg_weighted_year', 'model_firstreg_peak_year', 'model_firstreg_peak_count', 'model_firstreg_year_span', 'brand_firstreg_total_2014_2026', 'brand_firstreg_recent_share', 'brand_firstreg_old_share', 'brand_firstreg_weighted_year', 'brand_firstreg_peak_year', 'brand_firstreg_peak_count', 'brand_firstreg_year_span']

Missing registry lifecycle features:
[]

Found extended Traficom features:
['model_share_of_market', 'model_share_within_brand', 'model_share_over_10y', 'model_share_over_200k_km', 'model_automatic_share', 'model_petrol_share', 'model_diesel_share', 'model_ev_share', 'model_hybrid_share', 'model_turbo_share', 'brand_median_engine_cc', 'brand_median_power_kw', 'brand_median_mass_kg', 'brand_share_of_market', 'brand_share_over_10y', 'brand_share_over_200k_km', 'brand_automatic_share', 'brand_petrol_share', 'brand_diesel_share', 'brand_ev_s

,feature,status,reason
0,brand_is_known_model_family,missing_from_previous_manual_all,Low-cardinality metadata flag; slight validati...
1,mileage_missing_flag,missing_from_previous_manual_all,Tracks rows where mileage was originally missi...
2,first_seen_date,missing_from_previous_manual_all,Observed listing start within the crawl window...
3,last_seen_date,missing_from_previous_manual_all,Observed listing end within the crawl window; ...
4,brand_merge_key,recommended_exclusion,Redundant normalized brand key that overlaps w...
5,model_family_clean,recommended_exclusion,Collapsed model family duplicate of the existi...
6,model_looks_like_part_taxonomy,recommended_exclusion,"Constant in the training split, so it adds no ..."
7,model_merge_key,recommended_exclusion,Redundant normalized model key that overlaps w...
8,product_id,recommended_exclusion,High-cardinality listing identifier; encourage...
9,scrape_date,recommended_exclusion,Current crawl wave rather than part value; wor...


## 5. Select baseline feature set

In [7]:
features_baseline_only = baseline_features

assert len(features_baseline_only) > 0, "Add at least one column to baseline_features before training."

print("Number of baseline features:", len(features_baseline_only))
features_baseline_only

Number of baseline features: 13


['part_name',
 'quality_grade',
 'oem_number',
 'mileage',
 'brand',
 'model',
 'category',
 'subcategory',
 'year_start',
 'year_end',
 'year_span',
 'year_mid',
 'repair_status']

## 6. Build X and y

In [8]:
missing_features = [column for column in features_baseline_only if column not in train_df.columns]
assert len(missing_features) == 0, f"These selected features are missing from the dataset: {missing_features}"

X_train = train_df[features_baseline_only].copy()
X_validation = validation_df[features_baseline_only].copy()

y_train = train_df["price"].copy()
y_validation = validation_df["price"].copy()

## 7. Log-transform target

In [9]:
y_train_log = np.log(y_train)

y_train_log.head()

0    5.179534
1    5.179534
2    5.179534
3    5.179534
4    3.165475
Name: price, dtype: float64

## 8. Detect column types

In [10]:
numeric_features = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

numeric_features

['mileage', 'year_start', 'year_end', 'year_span', 'year_mid']

## 8. Detect column types

In [11]:
numeric_features = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

In [12]:
if "numeric_features" not in globals() or "categorical_features" not in globals():
    numeric_features = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
    categorical_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

column_type_summary = pd.DataFrame(
    {
        "column_type": ["numeric", "categorical"],
        "count": [len(numeric_features), len(categorical_features)],
        "columns": [numeric_features, categorical_features],
    }
)

display(column_type_summary)

,column_type,count,columns
0,numeric,5,"[mileage, year_start, year_end, year_span, yea..."
1,categorical,8,"[part_name, quality_grade, oem_number, brand, ..."


## 9. Tune Linear Model Pipeline

In [13]:
def build_linear_model_pipeline(X_train_current, estimator, onehot_min_frequency=5):
    numeric_features_current = X_train_current.select_dtypes(
        include=["number", "bool"]
    ).columns.tolist()
    categorical_features_current = X_train_current.select_dtypes(
        include=["object", "category"]
    ).columns.tolist()

    numeric_pipeline_current = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    categorical_pipeline_current = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=onehot_min_frequency)),
    ])

    preprocessor_current = ColumnTransformer(transformers=[
        ("num", numeric_pipeline_current, numeric_features_current),
        ("cat", categorical_pipeline_current, categorical_features_current),
    ])

    return Pipeline(steps=[
        ("preprocessor", preprocessor_current),
        ("model", clone(estimator)),
    ])


linear_model_candidates = {
    "ols_reference": {
        "estimator": LinearRegression(),
        "onehot_min_frequency": 5,
        "notes": "Original notebook setup for comparison.",
    },
    "ridge_alpha_0_05": {
        "estimator": Ridge(alpha=0.05),
        "onehot_min_frequency": 5,
        "notes": "Light regularization for sparse one-hot features.",
    },
    "ridge_alpha_0_1": {
        "estimator": Ridge(alpha=0.1),
        "onehot_min_frequency": 5,
        "notes": "Best validation MAE in the targeted rerun.",
    },
    "ridge_alpha_0_2": {
        "estimator": Ridge(alpha=0.2),
        "onehot_min_frequency": 5,
        "notes": "Slightly stronger regularization check.",
    },
}

tuning_features = recommended_model_features_leakage_safe
X_train_tuning = train_df[tuning_features].copy()
X_validation_tuning = validation_df[tuning_features].copy()

tuning_results = []
for config_name, config in linear_model_candidates.items():
    model_current = build_linear_model_pipeline(
        X_train_tuning,
        estimator=config["estimator"],
        onehot_min_frequency=config["onehot_min_frequency"],
    )
    model_current.fit(X_train_tuning, y_train_log)

    validation_pred_log_current = model_current.predict(X_validation_tuning)
    validation_pred_current = np.exp(validation_pred_log_current)

    tuning_results.append({
        "config": config_name,
        "trusted_for_selection": True,
        "model": type(config["estimator"]).__name__,
        "onehot_min_frequency": config["onehot_min_frequency"],
        "validation_MAE": mean_absolute_error(y_validation, validation_pred_current),
        "validation_RMSE": np.sqrt(mean_squared_error(y_validation, validation_pred_current)),
        "validation_R2": r2_score(y_validation, validation_pred_current),
        "notes": config["notes"],
    })

tuning_results_df = pd.DataFrame(tuning_results).sort_values(
    ["validation_MAE", "validation_RMSE"]
).reset_index(drop=True)

selected_linear_config_name = tuning_results_df.iloc[0]["config"]
selected_linear_config = linear_model_candidates[selected_linear_config_name]

print("Selected linear config:", selected_linear_config_name)
print(selected_linear_config)

display(
    tuning_results_df.style.format({
        "validation_MAE": "{:.4f}",
        "validation_RMSE": "{:.4f}",
        "validation_R2": "{:.4f}",
    })
)

Selected linear config: ridge_alpha_0_1
{'estimator': Ridge(alpha=0.1), 'onehot_min_frequency': 5, 'notes': 'Best validation MAE in the targeted rerun.'}


,config,model,onehot_min_frequency,validation_MAE,validation_RMSE,validation_R2,notes
0,ridge_alpha_0_1,Ridge,5,36.8083,158.9356,0.9214,Best validation MAE in the targeted rerun.
1,ridge_alpha_0_05,Ridge,5,36.9516,158.6810,0.9217,Light regularization for sparse one-hot features.
2,ridge_alpha_0_2,Ridge,5,37.0327,160.2908,0.9201,Slightly stronger regularization check.
3,ols_reference,LinearRegression,5,37.4254,158.8210,0.9215,Original notebook setup for comparison.


In [14]:
baseline_model = build_linear_model_pipeline(
    X_train,
    estimator=selected_linear_config["estimator"],
    onehot_min_frequency=selected_linear_config["onehot_min_frequency"],
)

selected_linear_config

{'estimator': Ridge(alpha=0.1),
 'onehot_min_frequency': 5,
 'notes': 'Best validation MAE in the targeted rerun.'}

## 10. Train Tuned Baseline Linear Model

In [15]:
baseline_model.fit(X_train, y_train_log)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers co

## 11. Predict and convert back to euro scale

In [16]:
validation_pred_log = baseline_model.predict(X_validation)

In [17]:
validation_pred = np.exp(validation_pred_log)

## 12. Preview Baseline Predictions


In [18]:
baseline_prediction_preview = pd.DataFrame({
    "actual_price": y_validation,
    "predicted_price": validation_pred,
})

baseline_prediction_preview.head()

,actual_price,predicted_price
0,177.6,172.135541
1,473.6,428.390580
2,142.1,138.702144
3,82.9,109.413156
4,177.6,96.233901


## 13. Baseline + Traficom features

This section tests whether external Traficom enrichment improves prediction compared to the listing-only baseline.

In [19]:
features_baseline_plus_traficom = baseline_features + traficom_features

print("Number of baseline + Traficom features:", len(features_baseline_plus_traficom))
features_baseline_plus_traficom

Number of baseline + Traficom features: 26


['part_name',
 'quality_grade',
 'oem_number',
 'mileage',
 'brand',
 'model',
 'category',
 'subcategory',
 'year_start',
 'year_end',
 'year_span',
 'year_mid',
 'repair_status',
 'model_total_registered',
 'model_median_vehicle_age',
 'model_mean_vehicle_age',
 'model_median_mileage',
 'model_mean_mileage',
 'model_median_engine_cc',
 'model_median_power_kw',
 'model_median_mass_kg',
 'brand_total_registered',
 'brand_median_vehicle_age',
 'brand_mean_vehicle_age',
 'brand_median_mileage',
 'brand_mean_mileage']

## 14. Build X and y for baseline + Traficom

In [20]:
missing_traficom_features = [
    column for column in features_baseline_plus_traficom if column not in train_df.columns
]
assert len(missing_traficom_features) == 0, (
    f"These selected features are missing from the dataset: {missing_traficom_features}"
)

X_train_traficom = train_df[features_baseline_plus_traficom].copy()
X_validation_traficom = validation_df[features_baseline_plus_traficom].copy()

## 15. Detect column types again

In [21]:
numeric_features_traficom = X_train_traficom.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features_traficom = X_train_traficom.select_dtypes(include=["object", "category"]).columns.tolist()

In [22]:
print("Numeric columns:", numeric_features_traficom)
print("\nCategorical columns:", categorical_features_traficom)

Numeric columns: ['mileage', 'year_start', 'year_end', 'year_span', 'year_mid', 'model_total_registered', 'model_median_vehicle_age', 'model_mean_vehicle_age', 'model_median_mileage', 'model_mean_mileage', 'model_median_engine_cc', 'model_median_power_kw', 'model_median_mass_kg', 'brand_total_registered', 'brand_median_vehicle_age', 'brand_mean_vehicle_age', 'brand_median_mileage', 'brand_mean_mileage']

Categorical columns: ['part_name', 'quality_grade', 'oem_number', 'brand', 'model', 'category', 'subcategory', 'repair_status']


## 16. Build A Fresh Tuned Linear-Model Pipeline

In [23]:
print("Using the selected linear config for baseline + Traficom:")
print(selected_linear_config_name)
selected_linear_config

Using the selected linear config for baseline + Traficom:
ridge_alpha_0_1


{'estimator': Ridge(alpha=0.1),
 'onehot_min_frequency': 5,
 'notes': 'Best validation MAE in the targeted rerun.'}

In [24]:
linear_regression_traficom = build_linear_model_pipeline(
    X_train_traficom,
    estimator=selected_linear_config["estimator"],
    onehot_min_frequency=selected_linear_config["onehot_min_frequency"],
)

## 17. Train baseline + Traficom model

In [25]:
linear_regression_traficom.fit(X_train_traficom, y_train_log)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers co

## 18. Predict on validation and convert back to euro scale

In [26]:
validation_pred_log_traficom = linear_regression_traficom.predict(X_validation_traficom)

In [27]:
validation_pred_traficom = np.exp(validation_pred_log_traficom)

## 19. Preview Baseline + Traficom Predictions

In [28]:
traficom_prediction_preview = pd.DataFrame({
    "actual_price": y_validation,
    "predicted_price": validation_pred_traficom,
})

traficom_prediction_preview.head()

,actual_price,predicted_price
0,177.6,172.118696
1,473.6,428.340957
2,142.1,138.696270
3,82.9,109.406392
4,177.6,96.255138


In [29]:
traficom_prediction_preview.describe()

,actual_price,predicted_price
count,1689.000000,1689.000000
mean,258.793428,247.391677
std,567.153248,530.281793
min,5.900000,8.310800
25%,59.000000,55.899306
50%,100.600000,98.113279
75%,236.000000,229.686185
max,4284.000000,4045.083722


## 20. All Recommended Features

This section trains the linear model on every recommended feature: the full manual feature union plus `first_seen_date`, `last_seen_date`, and `brand_is_known_model_family`, while still excluding IDs, duplicate keys, and the constant taxonomy flag.

In [30]:
features_all = recommended_model_features_leakage_safe

print("Number of trusted recommended model features:", len(features_all))
features_all

Number of recommended model features: 72


['part_name',
 'quality_grade',
 'oem_number',
 'mileage',
 'brand',
 'model',
 'category',
 'subcategory',
 'year_start',
 'year_end',
 'year_span',
 'year_mid',
 'repair_status',
 'brand_is_known_model_family',
 'model_total_registered',
 'model_median_vehicle_age',
 'model_mean_vehicle_age',
 'model_median_mileage',
 'model_mean_mileage',
 'model_median_engine_cc',
 'model_median_power_kw',
 'model_median_mass_kg',
 'model_share_of_market',
 'model_share_within_brand',
 'model_share_over_10y',
 'model_share_over_200k_km',
 'model_automatic_share',
 'model_petrol_share',
 'model_diesel_share',
 'model_ev_share',
 'model_hybrid_share',
 'model_turbo_share',
 'model_firstreg_total_2014_2026',
 'model_firstreg_year_span',
 'model_firstreg_peak_year',
 'model_firstreg_peak_count',
 'model_firstreg_recent_share',
 'model_firstreg_old_share',
 'model_firstreg_weighted_year',
 'brand_total_registered',
 'brand_median_vehicle_age',
 'brand_mean_vehicle_age',
 'brand_median_mileage',
 'brand_

## 21. Build X and y for all recommended features

In [31]:
missing_all_features = [column for column in features_all if column not in train_df.columns]
assert len(missing_all_features) == 0, (
    f"These selected features are missing from the dataset: {missing_all_features}"
)

X_train_all = train_df[features_all].copy()
X_validation_all = validation_df[features_all].copy()

## 22. Detect column types again

In [32]:
numeric_features_all = X_train_all.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features_all = X_train_all.select_dtypes(include=["object", "category"]).columns.tolist()

In [33]:
print("Numeric columns:", numeric_features_all)
print("\nCategorical columns:", categorical_features_all)

Numeric columns: ['mileage', 'year_start', 'year_end', 'year_span', 'year_mid', 'brand_is_known_model_family', 'model_total_registered', 'model_median_vehicle_age', 'model_mean_vehicle_age', 'model_median_mileage', 'model_mean_mileage', 'model_median_engine_cc', 'model_median_power_kw', 'model_median_mass_kg', 'model_share_of_market', 'model_share_within_brand', 'model_share_over_10y', 'model_share_over_200k_km', 'model_automatic_share', 'model_petrol_share', 'model_diesel_share', 'model_ev_share', 'model_hybrid_share', 'model_turbo_share', 'model_firstreg_total_2014_2026', 'model_firstreg_year_span', 'model_firstreg_peak_year', 'model_firstreg_peak_count', 'model_firstreg_recent_share', 'model_firstreg_old_share', 'model_firstreg_weighted_year', 'brand_total_registered', 'brand_median_vehicle_age', 'brand_mean_vehicle_age', 'brand_median_mileage', 'brand_mean_mileage', 'brand_median_engine_cc', 'brand_median_power_kw', 'brand_median_mass_kg', 'brand_share_of_market', 'brand_share_over

## 23. Build A Fresh Tuned Linear-Model Pipeline

In [34]:
print("Using the selected linear config for the recommended-feature model:")
print(selected_linear_config_name)
selected_linear_config

Using the selected linear config for the recommended-feature model:
ridge_alpha_0_1


{'estimator': Ridge(alpha=0.1),
 'onehot_min_frequency': 5,
 'notes': 'Best validation MAE in the targeted rerun.'}

In [35]:
linear_regression_all = build_linear_model_pipeline(
    X_train_all,
    estimator=selected_linear_config["estimator"],
    onehot_min_frequency=selected_linear_config["onehot_min_frequency"],
)

## 24. Train baseline + Traficom + registry lifecycle + listing dynamics model

In [36]:
linear_regression_all.fit(X_train_all, y_train_log)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers co

## 25. Predict on validation and convert back to euro scale

In [37]:
validation_pred_log_all = linear_regression_all.predict(X_validation_all)

In [38]:
validation_pred_all = np.exp(validation_pred_log_all)

## 26. Preview Recommended-Feature Predictions

These cells only preview the validation predictions in euro scale for the recommended-feature model. Formal model-comparison metrics and grouped diagnostics are reported in the later validation sections.

In [39]:
# Pair each validation observation with its prediction in euro scale.
recommended_prediction_preview = pd.DataFrame({
    "actual_price": y_validation,
    "predicted_price": validation_pred_all,
})

recommended_prediction_preview.head()

,actual_price,predicted_price
0,177.6,178.950189
1,473.6,438.098954
2,142.1,142.118570
3,82.9,101.254398
4,177.6,97.749068


In [40]:
# A quick descriptive summary helps check the prediction range before detailed validation.
recommended_prediction_preview.describe()

,actual_price,predicted_price
count,1689.000000,1689.000000
mean,258.793428,251.057554
std,567.153248,558.913532
min,5.900000,12.143369
25%,59.000000,54.060147
50%,100.600000,99.711354
75%,236.000000,228.795496
max,4284.000000,6541.716599


## 27. Validation Error Analysis By Part Group

This section evaluates validation-set errors for the existing linear regression model by part grouping. It reuses the available validation predictions and reports which groups are predicted most accurately and least accurately.

In [41]:
# Reuse the most detailed validation predictions already available in the notebook.
if "best_validation_predictions" in globals():
    validation_predictions_eur = best_validation_predictions
elif "validation_pred_all" in globals():
    validation_predictions_eur = validation_pred_all
elif "validation_pred_traficom" in globals():
    validation_predictions_eur = validation_pred_traficom
elif "validation_pred" in globals():
    validation_predictions_eur = validation_pred
else:
    raise NameError("No validation predictions were found in the notebook.")

# Build one validation results table in euro scale for grouped error analysis.
validation_results_df = validation_df[["price", "category", "subcategory", "part_name"]].copy()
validation_results_df = validation_results_df.rename(columns={"price": "actual_price"})
validation_results_df["predicted_price"] = validation_predictions_eur
validation_results_df["absolute_error"] = (
    validation_results_df["actual_price"]
    - validation_results_df["predicted_price"]
).abs()
validation_results_df["squared_error"] = (
    validation_results_df["actual_price"]
    - validation_results_df["predicted_price"]
) ** 2

validation_results_df.head()

,actual_price,category,subcategory,part_name,predicted_price,absolute_error,squared_error
0,177.6,airbag,contact roll airbag,"Contact roll Airbag - , e-",178.950189,1.350189,1.823010
1,473.6,airbag,passenger airbag,"Passenger airbag - , e-",438.098954,35.501046,1260.324289
2,142.1,airbag,left,"Curtain airbags - , e-(Left)",142.118570,0.018570,0.000345
3,82.9,airbag,seat assembled,"Seat assembled - , e-(Right front)",101.254398,18.354398,336.883914
4,177.6,airbag,all,"Side airbags - , e-(Right)",97.749068,79.850932,6376.171418


In [42]:
def summarize_group_errors(validation_results_df, group_column, min_count=20):
    summary = (
        validation_results_df
        .groupby(group_column, observed=False)
        .agg(
            count=("actual_price", "size"),
            MAE=("absolute_error", "mean"),
            median_absolute_error=("absolute_error", "median"),
            RMSE=("squared_error", lambda s: np.sqrt(s.mean())),
            within_10_eur=("absolute_error", lambda s: (s <= 10).mean()),
            within_20_eur=("absolute_error", lambda s: (s <= 20).mean()),
            within_50_eur=("absolute_error", lambda s: (s <= 50).mean()),
        )
        .reset_index()
    )

    summary = summary[summary["count"] >= min_count].copy()
    return summary.sort_values(["MAE", "RMSE", "count"]).reset_index(drop=True)


category_error_summary = summarize_group_errors(validation_results_df, "category")
subcategory_error_summary = summarize_group_errors(validation_results_df, "subcategory")
part_name_error_summary = summarize_group_errors(validation_results_df, "part_name")

## 28. Best And Worst Groups By MAE

The tables below summarize grouped validation errors in absolute euro terms. The `within_10_eur`, `within_20_eur`, and `within_50_eur` columns show how often predictions fall within practical absolute-error thresholds.

In [43]:
# Show the main absolute-error metrics used for grouped validation review.
summary_display_columns = [
    "count",
    "MAE",
    "median_absolute_error",
    "RMSE",
    "within_10_eur",
    "within_20_eur",
    "within_50_eur",
]

for summary_name, summary_df, group_label in [
    ("Category", category_error_summary, "category"),
    ("Subcategory", subcategory_error_summary, "subcategory"),
    ("Part name", part_name_error_summary, "part_name"),
]:
    print(f"\n{summary_name}: 10 best groups by MAE")
    display(
        summary_df.head(10).style.format({
            "MAE": "{:.2f}",
            "median_absolute_error": "{:.2f}",
            "RMSE": "{:.2f}",
            "within_10_eur": "{:.1%}",
            "within_20_eur": "{:.1%}",
            "within_50_eur": "{:.1%}",
        })
    )

    print(f"{summary_name}: 10 worst groups by MAE")
    display(
        summary_df.sort_values(["MAE", "RMSE", "count"], ascending=[False, False, False]).head(10).style.format({
            "MAE": "{:.2f}",
            "median_absolute_error": "{:.2f}",
            "RMSE": "{:.2f}",
            "within_10_eur": "{:.1%}",
            "within_20_eur": "{:.1%}",
            "within_50_eur": "{:.1%}",
        })
    )



Category: 10 best groups by MAE


,category,count,MAE,median_absolute_error,RMSE,within_10_eur,within_20_eur,within_50_eur
0,fuel,181,8.28,6.56,11.15,59.7%,95.6%,100.0%
1,electric / transmitter / databox / sensor,404,13.49,6.64,40.66,66.3%,85.1%,97.8%
2,vehicle exterior / suspension,294,17.04,11.31,29.95,46.6%,80.6%,94.2%
3,airbag,106,33.38,18.10,53.76,34.9%,54.7%,81.1%
4,gear box / drive axle / middle axle,241,40.90,21.16,89.79,31.5%,46.5%,83.4%
5,brakes,214,45.25,19.86,90.34,36.4%,50.5%,79.9%
6,engine,249,108.96,10.35,389.19,49.4%,73.1%,79.9%


Category: 10 worst groups by MAE


,category,count,MAE,median_absolute_error,RMSE,within_10_eur,within_20_eur,within_50_eur
6,engine,249,108.96,10.35,389.19,49.4%,73.1%,79.9%
5,brakes,214,45.25,19.86,90.34,36.4%,50.5%,79.9%
4,gear box / drive axle / middle axle,241,40.90,21.16,89.79,31.5%,46.5%,83.4%
3,airbag,106,33.38,18.10,53.76,34.9%,54.7%,81.1%
2,vehicle exterior / suspension,294,17.04,11.31,29.95,46.6%,80.6%,94.2%
1,electric / transmitter / databox / sensor,404,13.49,6.64,40.66,66.3%,85.1%,97.8%
0,fuel,181,8.28,6.56,11.15,59.7%,95.6%,100.0%



Subcategory: 10 best groups by MAE


...

,subcategory,count,MAE,median_absolute_error,RMSE,within_10_eur,within_20_eur,within_50_eur
23,engine gasoline,21,205.58,132.46,334.10,9.5%,23.8%,28.6%
22,passenger airbag,25,80.34,49.01,96.90,8.0%,8.0%,52.0%
21,right rear,49,25.68,23.59,34.11,12.2%,36.7%,91.8%
20,abs hydraulic pump,36,24.27,24.96,25.66,0.0%,33.3%,100.0%
19,all,153,21.16,12.49,33.58,47.7%,68.6%,88.2%
18,left front,39,20.18,13.79,28.81,35.9%,59.0%,84.6%
17,right front,48,18.63,12.14,24.42,39.6%,56.2%,95.8%
16,adaptiv farthållare sensor,24,18.40,16.55,19.19,0.0%,75.0%,100.0%
15,left rear,50,16.25,18.96,20.91,46.0%,58.0%,100.0%
14,airbag control unit,20,15.69,22.06,18.98,40.0%,40.0%,100.0%



Part name: 10 best groups by MAE


,part_name,count,MAE,median_absolute_error,RMSE,within_10_eur,within_20_eur,within_50_eur
0,Distributors Vacuum regulator -,20,5.22,4.44,5.80,90.0%,100.0%,100.0%
1,Brake Caliper -(Left front),20,5.88,1.47,16.86,95.0%,95.0%,95.0%
2,Trailing link rear -(Left),20,8.61,7.85,9.05,70.0%,100.0%,100.0%
3,Wheel bearing spindle shaft -(Right rear),20,22.43,20.87,24.85,0.0%,40.0%,100.0%
4,Drive shaft -(Right front),23,26.75,25.41,30.94,8.7%,30.4%,91.3%
5,ABS Hydraulic pump -,24,28.91,27.06,29.55,0.0%,0.0%,100.0%
6,Wheel bearing spindle shaft -(Left rear),21,29.09,26.10,30.10,0.0%,23.8%,100.0%
7,Drive shaft -(Left front),32,29.11,25.83,33.76,3.1%,28.1%,81.2%


Part name: 10 worst groups by MAE


,part_name,count,MAE,median_absolute_error,RMSE,within_10_eur,within_20_eur,within_50_eur
7,Drive shaft -(Left front),32,29.11,25.83,33.76,3.1%,28.1%,81.2%
6,Wheel bearing spindle shaft -(Left rear),21,29.09,26.10,30.10,0.0%,23.8%,100.0%
5,ABS Hydraulic pump -,24,28.91,27.06,29.55,0.0%,0.0%,100.0%
4,Drive shaft -(Right front),23,26.75,25.41,30.94,8.7%,30.4%,91.3%
3,Wheel bearing spindle shaft -(Right rear),20,22.43,20.87,24.85,0.0%,40.0%,100.0%
2,Trailing link rear -(Left),20,8.61,7.85,9.05,70.0%,100.0%,100.0%
1,Brake Caliper -(Left front),20,5.88,1.47,16.86,95.0%,95.0%,95.0%
0,Distributors Vacuum regulator -,20,5.22,4.44,5.80,90.0%,100.0%,100.0%


## 27. Select The Best Linear-Model Feature Set

Reuse the tuned linear configuration selected earlier across every feature-set experiment so the comparison stays focused on feature value instead of estimator noise.

In [44]:
excluded_features = recommended_excluded_features

feature_sets = {
    "baseline only": {
        "features": baseline_features,
        "trusted_for_selection": True,
    },
    "baseline + traficom_core": {
        "features": baseline_features + traficom_features,
        "trusted_for_selection": True,
    },
    "baseline + traficom_core + registry_lifecycle": {
        "features": baseline_features + traficom_features + registry_lifecycle_features,
        "trusted_for_selection": True,
    },
    "baseline + traficom_core + registry_lifecycle + traficom_extended": {
        "features": (
            baseline_features
            + traficom_features
            + registry_lifecycle_features
            + traficom_extended_features
        ),
        "trusted_for_selection": True,
    },
    "trusted manual all-features set": {
        "features": manual_all_feature_groups_leakage_safe,
        "trusted_for_selection": True,
    },
    "trusted recommended features": {
        "features": recommended_model_features_leakage_safe,
        "trusted_for_selection": True,
    },
    "exploratory previous manual all-features set": {
        "features": manual_all_feature_groups,
        "trusted_for_selection": False,
    },
    "exploratory all recommended features": {
        "features": recommended_model_features,
        "trusted_for_selection": False,
    },
}

experiment_results = []
experiment_feature_details = []
experiment_predictions = {}

In [45]:
def prepare_experiment_features(feature_list, train_df, validation_df, excluded_features):
    requested_features = list(dict.fromkeys(feature_list))
    requested_features = [
        feature for feature in requested_features
        if feature not in excluded_features
    ]

    available_features = [
        feature for feature in requested_features
        if feature in train_df.columns
    ]
    missing_features = [
        feature for feature in requested_features
        if feature not in train_df.columns
    ]

    X_train_current = train_df[available_features].copy()
    X_validation_current = validation_df[available_features].copy()

    return (
        requested_features,
        available_features,
        missing_features,
        X_train_current,
        X_validation_current,
    )


def build_linear_regression_pipeline(
    X_train_current,
    estimator=None,
    onehot_min_frequency=None,
):
    estimator = selected_linear_config["estimator"] if estimator is None else estimator
    onehot_min_frequency = (
        selected_linear_config["onehot_min_frequency"]
        if onehot_min_frequency is None
        else onehot_min_frequency
    )

    return build_linear_model_pipeline(
        X_train_current,
        estimator=estimator,
        onehot_min_frequency=onehot_min_frequency,
    )

In [46]:
for model_name, feature_config in feature_sets.items():
    feature_list = feature_config["features"]
    (
        requested_features,
        available_features,
        missing_features,
        X_train_current,
        X_validation_current,
    ) = prepare_experiment_features(
        feature_list,
        train_df,
        validation_df,
        excluded_features,
    )

    model_current = build_linear_regression_pipeline(X_train_current)
    model_current.fit(X_train_current, y_train_log)

    validation_pred_log_current = model_current.predict(X_validation_current)
    validation_pred_current = np.exp(validation_pred_log_current)
    experiment_predictions[model_name] = validation_pred_current

    validation_mae_current = mean_absolute_error(
        y_validation,
        validation_pred_current,
    )
    validation_rmse_current = np.sqrt(
        mean_squared_error(y_validation, validation_pred_current)
    )
    validation_r2_current = r2_score(
        y_validation,
        validation_pred_current,
    )

    experiment_results.append({
        "experiment": model_name,
        "trusted_for_selection": feature_config["trusted_for_selection"],
        "raw_columns_requested": len(requested_features),
        "raw_columns_used": len(available_features),
        "raw_columns_missing": len(missing_features),
        "validation_MAE": validation_mae_current,
        "validation_RMSE": validation_rmse_current,
        "validation_R2": validation_r2_current,
    })

    experiment_feature_details.append({
        "experiment": model_name,
        "trusted_for_selection": feature_config["trusted_for_selection"],
        "used_features_count": len(available_features),
        "used_features_list": available_features,
        "missing_features_list": missing_features,
    })

## 28. Validate The Best Linear Regression Model
 MAE as the main validation metrics, backup R^2 and MSE, and extra MAPE. 

In [47]:
# Rebuild the best-model validation dataframe from the current experiment results.
if "experiment_results" not in globals() or len(experiment_results) == 0:
    raise NameError(
        "experiment_results is not defined. Run the feature-set comparison section first."
    )

experiment_results_df = pd.DataFrame(experiment_results)
if "trusted_for_selection" not in experiment_results_df.columns:
    experiment_results_df["trusted_for_selection"] = True

trusted_experiment_results_df = experiment_results_df[
    experiment_results_df["trusted_for_selection"]
].copy()
if trusted_experiment_results_df.empty:
    trusted_experiment_results_df = experiment_results_df.copy()

best_experiment_name = trusted_experiment_results_df.sort_values(
    ["validation_MAE", "validation_RMSE"]
).iloc[0]["experiment"]
best_validation_predictions = experiment_predictions[best_experiment_name]

best_model_validation_df = pd.DataFrame({
    "actual_price": y_validation,
    "predicted_price": best_validation_predictions,
})
best_model_validation_df["residual"] = (
    best_model_validation_df["actual_price"]
    - best_model_validation_df["predicted_price"]
)
best_model_validation_df["absolute_error"] = (
    best_model_validation_df["residual"].abs()
)
best_model_validation_df["squared_error"] = (
    best_model_validation_df["residual"] ** 2
)
best_model_validation_df["absolute_percentage_error"] = (
    best_model_validation_df["absolute_error"]
    / best_model_validation_df["actual_price"].clip(lower=1)
)
best_model_validation_df["prediction_direction"] = np.where(
    best_model_validation_df["residual"] >= 0,
    "underpredicted",
    "overpredicted",
)
best_model_validation_df["actual_price_band"] = pd.qcut(
    best_model_validation_df["actual_price"],
    q=5,
    duplicates="drop",
)

# Split the best-model validation errors by price band and prediction direction.
best_model_direction_summary = (
    best_model_validation_df
    .groupby(["actual_price_band", "prediction_direction"], observed=False)
    .agg(
        samples=("actual_price", "size"),
        mean_actual_price=("actual_price", "mean"),
        mean_predicted_price=("predicted_price", "mean"),
        mae=("absolute_error", "mean"),
    )
    .reset_index()
)

best_model_direction_summary.style.format({
    "mean_actual_price": "{:.1f}",
    "mean_predicted_price": "{:.1f}",
    "mae": "{:.2f}",
})

,actual_price_band,prediction_direction,samples,mean_actual_price,mean_predicted_price,mae
0,"(5.899, 47.4]",overpredicted,244,34.5,40.8,6.34
1,"(5.899, 47.4]",underpredicted,105,37.8,32.4,5.39
2,"(47.4, 82.6]",overpredicted,178,61.0,67.3,6.29
3,"(47.4, 82.6]",underpredicted,157,61.3,53.6,7.76
4,"(82.6, 154.06]",overpredicted,173,114.3,124.1,9.75
5,"(82.6, 154.06]",underpredicted,156,105.1,90.9,14.20
6,"(154.06, 248.42]",overpredicted,184,200.4,226.2,25.82
7,"(154.06, 248.42]",underpredicted,154,210.8,183.6,27.15
8,"(248.42, 4284.0]",overpredicted,129,925.5,1045.2,119.74
9,"(248.42, 4284.0]",underpredicted,209,858.1,717.3,140.84


In [48]:
# Build a full best-model validation table for absolute and percentage error diagnostics.
best_validation_predictions = experiment_predictions[best_experiment_name]

best_model_validation_df = pd.DataFrame({
    "actual_price": y_validation,
    "predicted_price": best_validation_predictions,
})
best_model_validation_df["residual"] = (
    best_model_validation_df["actual_price"]
    - best_model_validation_df["predicted_price"]
)
best_model_validation_df["absolute_error"] = (
    best_model_validation_df["residual"].abs()
)
best_model_validation_df["squared_error"] = (
    best_model_validation_df["residual"] ** 2
)
best_model_validation_df["absolute_percentage_error"] = (
    best_model_validation_df["absolute_error"]
    / best_model_validation_df["actual_price"].clip(lower=1)
)
best_model_validation_df["prediction_direction"] = np.where(
    best_model_validation_df["residual"] >= 0,
    "underpredicted",
    "overpredicted",
)
best_model_validation_df["actual_price_band"] = pd.qcut(
    best_model_validation_df["actual_price"],
    q=5,
    duplicates="drop",
)

# The price-band summary includes MAPE because percentage error is informative across very different price levels.
best_model_price_band_summary = (
    best_model_validation_df
    .groupby("actual_price_band", observed=False)
    .agg(
        samples=("actual_price", "size"),
        min_actual_price=("actual_price", "min"),
        max_actual_price=("actual_price", "max"),
        mean_actual_price=("actual_price", "mean"),
        mae=("absolute_error", "mean"),
        rmse=("squared_error", lambda s: np.sqrt(s.mean())),
        mape=("absolute_percentage_error", "mean"),
        median_residual=("residual", "median"),
    )
    .reset_index()
)

best_model_price_band_summary.style.format({
    "min_actual_price": "{:.1f}",
    "max_actual_price": "{:.1f}",
    "mean_actual_price": "{:.1f}",
    "mae": "{:.2f}",
    "rmse": "{:.2f}",
    "mape": "{:.1%}",
    "median_residual": "{:+.2f}",
})

,actual_price_band,samples,min_actual_price,max_actual_price,mean_actual_price,mae,rmse,mape,median_residual
0,"(5.899, 47.4]",349,5.9,47.4,35.5,6.05,8.75,19.8%,-1.93
1,"(47.4, 82.6]",335,47.6,82.6,61.1,6.98,10.51,11.3%,-0.56
2,"(82.6, 154.06]",329,82.8,153.9,109.9,11.86,16.83,10.9%,-0.33
3,"(154.06, 248.42]",338,154.1,248.3,205.1,26.43,41.45,13.5%,-1.36
4,"(248.42, 4284.0]",338,248.6,4284.0,883.9,132.79,352.20,13.9%,+22.88


In [49]:
experiment_results_df = pd.DataFrame(experiment_results)
if "trusted_for_selection" not in experiment_results_df.columns:
    experiment_results_df["trusted_for_selection"] = True

trusted_experiment_results_df = experiment_results_df[
    experiment_results_df["trusted_for_selection"]
].copy()
if trusted_experiment_results_df.empty:
    trusted_experiment_results_df = experiment_results_df.copy()

baseline_candidates_df = trusted_experiment_results_df.loc[
    trusted_experiment_results_df["experiment"] == "baseline only"
]
if baseline_candidates_df.empty:
    baseline_candidates_df = experiment_results_df.loc[
        experiment_results_df["experiment"] == "baseline only"
    ]
if baseline_candidates_df.empty:
    raise NameError("Could not find the 'baseline only' row in experiment_results.")

baseline_row = baseline_candidates_df.iloc[0]

experiment_results_df["delta_MAE_vs_baseline"] = (
    experiment_results_df["validation_MAE"]
    - baseline_row["validation_MAE"]
)
experiment_results_df["delta_RMSE_vs_baseline"] = (
    experiment_results_df["validation_RMSE"]
    - baseline_row["validation_RMSE"]
)
experiment_results_df["mae_rank"] = (
    experiment_results_df["validation_MAE"]
    .rank(method="dense")
    .astype(int)
)
experiment_results_df["rmse_rank"] = (
    experiment_results_df["validation_RMSE"]
    .rank(method="dense")
    .astype(int)
)

best_experiment_name = trusted_experiment_results_df.sort_values(
    ["validation_MAE", "validation_RMSE"]
).iloc[0]["experiment"]

display_columns = [
    "experiment",
    "trusted_for_selection",
    "raw_columns_used",
    "validation_MAE",
    "delta_MAE_vs_baseline",
    "mae_rank",
    "validation_RMSE",
    "delta_RMSE_vs_baseline",
    "rmse_rank",
    "validation_R2",
]

experiment_results_df.sort_values(
    ["trusted_for_selection", "validation_MAE", "validation_RMSE"],
    ascending=[False, True, True],
)[display_columns].style.format({
    "validation_MAE": "{:.4f}",
    "delta_MAE_vs_baseline": "{:+.4f}",
    "validation_RMSE": "{:.4f}",
    "delta_RMSE_vs_baseline": "{:+.4f}",
    "validation_R2": "{:.4f}",
})

,experiment,raw_columns_used,validation_MAE,delta_MAE_vs_baseline,mae_rank,validation_RMSE,delta_RMSE_vs_baseline,rmse_rank,validation_R2
5,all recommended features,72,36.8083,-5.7383,1,158.9356,+5.3493,5,0.9214
4,previous manual all-features set,68,38.1127,-4.4339,2,169.3642,+15.7779,6,0.9108
0,baseline only,13,42.5466,+0.0000,3,153.5862,+0.0000,1,0.9266
1,baseline + traficom_core,26,42.5496,+0.0030,4,153.6016,+0.0153,2,0.9266
3,baseline + traficom_core + registry_lifecycle + traficom_extended,62,42.5503,+0.0037,5,153.6047,+0.0185,3,0.9266
2,baseline + traficom_core + registry_lifecycle,40,42.6022,+0.0556,6,153.6604,+0.0741,4,0.9266
